In [1]:
from pydantic import BaseModel
from typing import Optional

class DailyReport(BaseModel):
    reasoning: str                        # always ask model to reason first
    Vendor: Optional[str] = None          # None if not found — no guessing
    Date: Optional[str] = None            # YYYY-MM-DD or None
    TotalWorkers: Optional[int] = None    # sum of Number column or None
    TotalWorkHr: Optional[float] = None   # total manhours or None

In [2]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

In [ ]:
SYSTEM_PROMPT = "You extract structured data from contractor daily work reports submitted TO Whiting-Turner."

EXTRACTION_PROMPT = """Extract from this daily report. If a field is not clearly present, return null — do NOT guess.

- Vendor: the SUBCONTRACTOR company submitting this report (NOT Whiting-Turner — they are the GC receiving it)
- Date: YYYY-MM-DD. Year is always 2024 or 2025
- TotalWorkers: SUM of the Number/Count column across all worker rows
- TotalWorkHr: printed Total Manhours if available, else sum (hours × workers) per row
"""

In [4]:
import base64
from pathlib import Path

def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [5]:
def extract_from_image(image_path: str):
    ext  = Path(image_path).suffix.lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"
    b64  = image_to_base64(image_path)

    try:
        response = client.beta.chat.completions.parse(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}", "detail": "high"}},
                    {"type": "text",      "text": EXTRACTION_PROMPT}
                ]}
            ],
            response_format=DailyReport,
            temperature=0
        )

        if response.choices[0].message.refusal:
            return None

        return response.choices[0].message.parsed

    except Exception as e:
        print(f"  ❌ {e}")
        return None

In [ ]:
import json

IMGS_FOLDER = "imgs"
OUTPUT_FILE = "res2.jsonl"

image_files = sorted(
    Path(IMGS_FOLDER).glob("img*.png"),
    key=lambda p: int(p.stem.replace("img", ""))
)

print(f"Found {len(image_files)} images\n")
print(f"{'File':<12} {'Vendor':<40} {'Date':<12} {'Workers':<10} {'Hours'}")
print("-" * 85)

with open(OUTPUT_FILE, "w") as out_f:
    for img_path in image_files:
        result = extract_from_image(str(img_path))

        if result:
            row = result.model_dump()
            row["source_file"] = img_path.name
            out_f.write(json.dumps(row) + "\n")
            out_f.flush()

            # Display all 4 fields cleanly
            print(f"{img_path.name:<12} {str(result.Vendor):<40} {str(result.Date):<12} {str(result.TotalWorkers):<10} {result.TotalWorkHr}")
        else:
            print(f"{img_path.name:<12} ⚠️  skipped")

print(f"\n✅ Saved to {OUTPUT_FILE}")

Found 13 images

File         Vendor                                   Date         Workers    Hours
-------------------------------------------------------------------------------------
img1.png     CSC                                      2025-02-17   4          32.0
img2.png     H.T.K masonry                            2024-12-17   4          32.0
img3.png     None                                     None         4          32.0
img4.png     None                                     2025-01-23   5          48.0
img5.png     None                                     None         6          33.0
img6.png     Tirone Electric Data                     2025-08-15   43         308.0
img7.png     None                                     2025-12-19   7          53.5
img8.png     Md fab                                   2025-09-17   4          32.0
img9.png     HJK                                      None         3          6.0
img10.png    Maryland Mechanical                      None        

In [7]:
SYSTEM_PROMPT = "You extract structured data from contractor daily work reports submitted TO Whiting-Turner."

EXTRACTION_PROMPT = """Extract from this daily report. If a field is not clearly present, return null — do NOT guess.

- Vendor: the SUBCONTRACTOR company submitting this report (NOT Whiting-Turner — they are the GC receiving it) search files and symbols or Key words which would have Vendor/Contractor company 
- Date: YYYY-MM-DD. Year is always greater then 2000
- TotalWorkers: SUM of the Number/Count column across all worker rows
- TotalWorkHr: printed Total Manhours if available, else sum (hours × workers) per row
"""

In [8]:
import json

IMGS_FOLDER = "imgs"
OUTPUT_FILE = "res4.jsonl"

image_files = sorted(
    Path(IMGS_FOLDER).glob("img*.png"),
    key=lambda p: int(p.stem.replace("img", ""))
)

print(f"Found {len(image_files)} images\n")
print(f"{'File':<12} {'Vendor':<40} {'Date':<12} {'Workers':<10} {'Hours'}")
print("-" * 85)

with open(OUTPUT_FILE, "w") as out_f:
    for img_path in image_files:
        result = extract_from_image(str(img_path))

        if result:
            row = result.model_dump()
            row["source_file"] = img_path.name
            out_f.write(json.dumps(row) + "\n")
            out_f.flush()

            # Display all 4 fields cleanly
            print(f"{img_path.name:<12} {str(result.Vendor):<40} {str(result.Date):<12} {str(result.TotalWorkers):<10} {result.TotalWorkHr}")
        else:
            print(f"{img_path.name:<12} ⚠️  skipped")

print(f"\n✅ Saved to {OUTPUT_FILE}")

Found 13 images

File         Vendor                                   Date         Workers    Hours
-------------------------------------------------------------------------------------
img1.png     CSC                                      2025-02-17   4          32.0
img2.png     H.J.K masonry                            2024-12-17   4          32.0
img3.png     Corrado                                  2023-10-02   4          32.0
img4.png     None                                     2025-01-23   5          7646.0
img5.png     Wolfe Contractors, Inc.                  2026-03-12   6          33.0
img6.png     Tirone Electric Data                     2025-08-11   43         301.0
img7.png     None                                     2025-12-19   7          53.5
img8.png     Md fab                                   2025-09-17   4          32.0
img9.png     HSK                                      None         3          6.0
img10.png    Maryland Mechanical                      2026-02-25

In [9]:
from pdf2image import convert_from_path
from PIL import Image

PDF_PATH = "pdfs\1.20.25-1.24.25.pdf"   # change to your PDF

# convert all pages to PIL images
pages = convert_from_path(PDF_PATH, dpi=200)

print(f"Total pages: {len(pages)}")

# preview first page size
print(f"Page 1 size: {pages[0].size}")

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [10]:
import fitz   # PyMuPDF
import base64
from io import BytesIO
from PIL import Image

def pdf_to_images_base64(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    images_b64 = []
    
    for page in doc:
        # render page to image
        pix = page.get_pixmap(dpi=200)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        # convert to base64
        buf = BytesIO()
        img.save(buf, format="JPEG")
        images_b64.append(base64.b64encode(buf.getvalue()).decode("utf-8"))
    
    doc.close()
    return images_b64

pages_b64 = pdf_to_images_base64("pdfs/1.20.25-1.24.25.pdf")
print(f"Total pages: {len(pages_b64)} ✅")

Total pages: 5 ✅


In [11]:
from pydantic import BaseModel
from typing import Optional

class DailyReport(BaseModel):
    reasoning: str
    Vendor: str
    Date: str
    TotalWorkers: int
    TotalWorkHr: float

class DailyReportList(BaseModel):
    reports: list[DailyReport]  # one per page/report found in PDF

In [15]:
import fitz
import base64, os, json
from io import BytesIO
from PIL import Image
from pathlib import Path

PDF_FOLDER = "pdfs"

def pdf_to_images_b64(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    images = []
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        buf = BytesIO()
        img.save(buf, format="JPEG")
        images.append(base64.b64encode(buf.getvalue()).decode("utf-8"))
    doc.close()
    return images

def extract_from_pdf(pdf_path: str) -> dict:
    print(f"\nProcessing: {pdf_path}")
    
    images_b64 = pdf_to_images_b64(pdf_path)
    print(f"  Pages: {len(images_b64)}")
    
    # build content — all pages as images
    content = []
    for b64 in images_b64:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{b64}",
                "detail": "high"
            }
        })
    
    # add instruction at end
    content.append({
        "type": "text",
        "text": """This PDF has multiple pages. Each page is one daily report or multiple pages might have information spread across. Your work is to analyse whole file and give accurate info on below fields
Extract one record per page:
- Vendor: contractor company name from whom WHiting Turner is taking service from
- Date: YYYY-MM-DD. Year is always greater then 2000
- TotalWorkers: SUM of the Number column across all rows on that page
- TotalWorkHr: use printed Total Manhours if filled, else sum hours x workers
First write your reasoning, then extract values."""
    })
    
    response = client.beta.chat.completions.parse(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        messages=[
            {"role": "system", "content": "You extract structured data from contractor daily work report images."},
            {"role": "user", "content": content}
        ],
        response_format=DailyReportList,
        temperature=0
    )
    
    result = response.choices[0].message.parsed
    
    print(f"  Reports found: {len(result.reports)}")
    print(f"  Tokens: input={response.usage.prompt_tokens}, output={response.usage.completion_tokens}")
    
    return {
        "file": Path(pdf_path).name,
        "pages": len(images_b64),
        "reports": result.model_dump(exclude={"reports": {"__all__": {"reasoning"}}})["reports"],
        "reports_with_reasoning": result.model_dump()["reports"],  # keep for quality check
        "tokens_used": response.usage.total_tokens
    }

In [ ]:
all_results = []
errors = []

pdf_files = list(Path(PDF_FOLDER).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs to process")

for pdf_path in pdf_files:
    try:
        result = extract_from_pdf(str(pdf_path))
        all_results.append(result)
    except Exception as e:
        print(f"  ❌ Error: {e}")
        errors.append({"file": pdf_path.name, "error": str(e)})

# save to JSON
with open("extraction_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n✅ Done — {len(all_results)} PDFs processed, {len(errors)} errors")
print(f"Results saved to extraction_results.json")

Found 16 PDFs to process

Processing: pdfs\1.20.25-1.24.25.pdf
  Pages: 5
  Reports found: 5
  Tokens: input=6463, output=573

Processing: pdfs\2.17.25-2.18.25 Dailys.pdf
  Pages: 2
  Reports found: 2
  Tokens: input=2686, output=264

Processing: pdfs\24.12.16 HJK Dalies.pdf
  Pages: 4
  Reports found: 4
  Tokens: input=5100, output=410

Processing: pdfs\3.12.26 - Daily Report.pdf
  Pages: 4
  Reports found: 1
  Tokens: input=5256, output=125

Processing: pdfs\CCC - WT Daily UD ECUUP (4).pdf
  Pages: 1
  Reports found: 1
  Tokens: input=1518, output=179

Processing: pdfs\Daily Job Diary 1-23-25.pdf
  Pages: 1
  Reports found: 1
  Tokens: input=1518, output=143

Processing: pdfs\Daily Log - 03-12-2026 - MIH Tower & CUP (1).pdf
  Pages: 3
  Reports found: 1
  Tokens: input=4010, output=113

Processing: pdfs\DAILY REPORT NOVEN - Sub Example.pdf
  Pages: 1
  Reports found: 1
  Tokens: input=1518, output=209

Processing: pdfs\DR_12192025 (1).pdf
  Pages: 1
  Reports found: 1
  Tokens: input

In [16]:
# Print clean summary table
print(f"{'File':<40} {'Reports':<10} {'Tokens':<10}")
print("-" * 60)

total_tokens = 0
for r in all_results:
    print(f"{r['file']:<40} {len(r['reports']):<10} {r['tokens_used']:<10}")
    total_tokens += r['tokens_used']

print("-" * 60)
print(f"Total tokens used: {total_tokens}")
print(f"\n--- Sample reasoning (first report) ---")
if all_results and all_results[0]["reports_with_reasoning"]:
    print(all_results[0]["reports_with_reasoning"][0]["reasoning"])

File                                     Reports    Tokens    
------------------------------------------------------------
1.20.25-1.24.25.pdf                      5          6951      
2.17.25-2.18.25 Dailys.pdf               2          2940      
24.12.16 HJK Dalies.pdf                  4          5476      
3.12.26 - Daily Report.pdf               1          5355      
CCC - WT Daily UD ECUUP (4).pdf          1          1665      
Daily Job Diary 1-23-25.pdf              1          1676      
Daily Log - 03-12-2026 - MIH Tower & CUP (1).pdf 1          4115      
DAILY REPORT NOVEN - Sub Example.pdf     1          1697      
DR_12192025 (1).pdf                      1          1410      
f39f9ee9-980d-47c2-a70a-87290fab69d0.pdf 1          5472      
GMV Sub Daily Report.pdf                 2          2976      
HJK Masonry Daily Report 3_27_2025.pdf   1          1590      
MMSI Daily.pdf                           1          1681      
Nash Daily 6.25.25.pdf                   1       